In [1]:
import boto3

bucket_name = "dataminds-homeworks"
s3_file_key = "data_usage_production.parquet"  # e.g. 'folder/myfile.txt'
local_file_path = "data_usage_production.parquet"  # Local destination

# Create an S3 client (remove `bucket_name` here — not a valid argument for boto3.client)
s3 = boto3.client("s3", region_name="us-east-1")

# Download the file
try:
    s3.download_file(bucket_name, s3_file_key, local_file_path)
    print(
        f"✅ File downloaded successfully from s3://{bucket_name}/{s3_file_key} to {local_file_path}"
    )
except Exception as e:
    print("❌ Error downloading file:", e)

✅ File downloaded successfully from s3://dataminds-homeworks/data_usage_production.parquet to data_usage_production.parquet


In [ ]:
# %pip install category_encoders

In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from category_encoders import CatBoostEncoder

In [3]:
df = pd.read_parquet("data_usage_production.parquet")
df.set_index("telephone_number", inplace=True)

# I kept crashing the kernal because the ram wasn't enough so i did this
df_sample = df.sample(n=20000, random_state=42) if len(df) > 20000 else df

# Target log-transform was used
y = np.log1p(df_sample["data_compl_usg_local_m1"])
X = df_sample.drop("data_compl_usg_local_m1", axis=1)

# Numeric and Categorical feature lists
numeric_features = [
    "refill_total_m2",
    "refill_total_m3",
    "frequency",
    "recency",
    "tenure",
    "lastrefillamount_m2",
    "tot_inact_status_days_l1m_m2",
    "refill_total_m4",
    "data_compl_usg_local_m2",
    "data_compl_usg_local_m3",
    "data_compl_usg_local_m4",
]
categorical_features = X.select_dtypes(include="object").columns.tolist()

X = X[numeric_features + categorical_features]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[("encoder", CatBoostEncoder(cols=categorical_features, random_state=42))]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

clf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(n_estimators=20, max_depth=10, random_state=42, n_jobs=-1),
        ),
    ]
)

# the fitting and predicting
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Model R² Score: %.3f" % r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

y_pred_original = np.expm1(y_pred)

Model R² Score: 0.745
RMSE: 1.8160811807551127


In [ ]:
import boto3

# Replace with your actual credentials and info
bucket_name = "dataminds-homeworks"
s3_file_key = "ali-gasimov-fe1.ipynb"
local_file_path = "ali-gasimov-fe1.ipynb"

# Create an S3 client
s3 = boto3.client("s3")

# Upload the file
try:
    s3.upload_file(local_file_path, bucket_name, s3_file_key)
    print(f"File uploaded successfully to s3://{bucket_name}/{s3_file_key}")
except Exception as e:
    print("Error uploading file:", e)